In [ ]:
# =====================================================================
# BƯỚC 1: CÀI ĐẶT THƯ VIỆN CHUYỂN ĐỔI ONNX CHUẨN TRÊN COLAB
# =====================================================================
!pip install tf2onnx onnx

import os
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from google.colab import files
import tf2onnx

# =====================================================================
# BƯỚC 2: ĐỌC VÀ TIỀN XỬ LÝ DỮ LIỆU
# =====================================================================
print("--- Đang nạp dữ liệu từ file Dataset.csv tải lên trực tiếp ---")
if not os.path.exists("Dataset.csv"):
    raise FileNotFoundError("Vui lòng upload file 'Dataset.csv' lên thư mục gốc của Colab trước!")

df = pd.read_csv("Dataset.csv")

# Bóc tách thuộc tính đặc trưng (X) và nhãn (y) - Tự động nhận diện 16 cột cảm biến
X = df.drop(columns=['label']).values.astype(np.float32)
y = df['label'].values.astype(np.float32)

# Chuẩn hóa dữ liệu theo phân phối chuẩn Z-score
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=42
)

# =====================================================================
# BƯỚC 3: KIẾN TRÚC MẠNG DEEP MLP CHUẨN FUNCTIONAL API
# =====================================================================
input_shape = X_train.shape[1]
inputs = Input(shape=(input_shape,), dtype=tf.float32, name="float_input")

x = Dense(128, activation='relu')(inputs)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)

x = Dense(64, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.2)(x)

x = Dense(32, activation='relu')(x)
outputs = Dense(1, activation='sigmoid', name="output_layer")(x)

model = Model(inputs=inputs, outputs=outputs, name="WaterLog_DeepMLP")

# =====================================================================
# BƯỚC 4: BIÊN DỊCH VÀ HUẤN LUYỆN HIỆU NĂNG CAO
# =====================================================================
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("\n--- BẮT ĐẦU HUẤN LUYỆN DEEP MULTILAYER PERCEPTRON ---")
model.fit(
    X_train, y_train,
    epochs=15,
    batch_size=512,
    validation_data=(X_test, y_test),
    verbose=1
)

# =====================================================================
# BƯỚC 5: XUẤT ĐÓNG GÓI MÔ HÌNH SANG FILE ONNX VÀ CẬP NHẬT TOÁN HỌC TỐI ƯU
# =====================================================================
print("\n--- Đang thực hiện chuyển đổi và đóng gói đồ thị sang file ONNX ---")
spec = (tf.TensorSpec((None, input_shape), tf.float32, name="float_input"),)

@tf.function
def to_onnx(x):
    return model(x)

model_proto, _ = tf2onnx.convert.from_function(
    to_onnx,
    input_signature=spec,
    output_path="deep_mlp_model.onnx"
)

# TỐI ƯU TOÁN HỌC: Xuất trực tiếp Độ lệch chuẩn (Standard Deviation) thay vì Phương sai
# scaler.scale_ chính bằng căn bậc hai của scaler.var_ (đã được tính toán tối ưu bằng C++ của sklearn)
np.savetxt("scaler_mean.txt", scaler.mean_)
np.savetxt("scaler_std.txt", scaler.scale_) # Đổi tên file thành scaler_std.txt để tường minh ý nghĩa khoa học

print("[+] Đã sinh xong 3 file sạch: deep_mlp_model.onnx, scaler_mean.txt, scaler_std.txt")

# =====================================================================
# BƯỚC 6: TỰ ĐỘNG TẢI FILE KẾT QUẢ VỀ MÁY TÍNH CÁ NHÂN
# =====================================================================
print("\n--- Hệ thống tự động kích hoạt tải 3 file kết quả về máy tính... ---")
download_files = ["deep_mlp_model.onnx", "scaler_mean.txt", "scaler_std.txt"]

for file_name in download_files:
    if os.path.exists(file_name):
        print(f"[Tải xuống] -> {file_name}")
        files.download(file_name)

--- Đang nạp dữ liệu từ file Dataset.csv tải lên trực tiếp ---

--- BẮT ĐẦU HUẤN LUYỆN DEEP MULTILAYER PERCEPTRON ---
Epoch 1/15
282/282 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - accuracy: 0.9462 - loss: 0.1429 - val_accuracy: 0.9631 - val_loss: 0.0907
Epoch 2/15
282/282 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9637 - loss: 0.0906 - val_accuracy: 0.9671 - val_loss: 0.0695
Epoch 3/15
282/282 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9722 - loss: 0.0673 - val_accuracy: 0.9701 - val_loss: 0.0629
Epoch 4/15
282/282 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9807 - loss: 0.0481 - val_accuracy: 0.9816 - val_loss: 0.0400
Epoch 5/15
282/282 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9848 - loss: 0.0379 - val_accuracy: 0.9907 - val_loss: 0.0221
Epoch 6/15
282/282 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9875 - loss: 0.0320 - val_accuracy: 0.9933 - val_loss: 0.0167
Epoch 7/15
282/282 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9888 - loss: 0.0286 - val_accuracy: 0.9926 - va

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[Tải xuống] -> scaler_mean.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[Tải xuống] -> scaler_std.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>